# 10장 실습 ① — 순환 신경망은 얼마나 불안정한가

**PyTorch 판**

> ⚠ **이 노트북은 교과서적 결론이 나오지 않습니다.**
>
> *"LSTM이 기울기 소실을 해결한다"* 는 이 실험으로 재현되지 않았습니다.
> 대신 훨씬 중요한 것이 나옵니다 — **학습이 지독하게 불안정합니다.**

실험 ①만 보고 결론 내리지 마십시오. ②와 ③까지 보셔야 합니다.

## 10.0 준비

In [ ]:
try:
    import dlbook
except ImportError:
    !pip install -q "dlbook @ git+https://github.com/dhrim/deep-learning-in-one-semester.git"
    import dlbook

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import dlbook
from dlbook import data, metrics, plot

dlbook.set_seed(42)
plot.use_korean()
print(dlbook.versions())

## 10.1 실험대 — 표시된 자리를 기억하기

순차열은 대부분 잡음입니다. **한 자리**에만 신호가 있고, 두 번째 채널이
그 자리를 표시합니다. **그 신호의 부호**를 맞힙니다.

표시된 자리가 앞쪽이므로, 그 값을 **끝까지 여러 걸음 들고 가야** 합니다.

In [ ]:
# 표시된 자리의 값을 끝까지 들고 가야 풀리는 문제.
x, y = data.memory_task(4000, length=80, seed=42)
sl = data.split(x, y, val_ratio=0.15, test_ratio=0.15, seed=42)
print(sl.summary())

fig, ax = plt.subplots(figsize=(8.5, 2.6))
ax.plot(x[0, :, 0], lw=1.2, label="값")
ax.plot(x[0, :, 1], lw=1.2, ls="--", label="표시")
ax.set_xlabel("걸음"); ax.legend(fontsize=9); ax.grid(alpha=0.3)
ax.set_title(f"표시된 자리의 부호를 맞혀야 합니다 (정답 {y[0]})")
plt.show()

## 10.2 학습 함수 — 여기만 판마다 다릅니다

**PyTorch 판에 `Recurrent` 래퍼가 하나 더 있는 것**에 주목하십시오.
PyTorch의 순환 층은 (출력 전체, 마지막 상태)를 돌려주므로,
Keras의 기본 동작(마지막 것만)과 맞추려면 감싸야 합니다.

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

device = "cuda" if torch.cuda.is_available() else "cpu"

class Recurrent(nn.Module):
    """마지막 걸음의 상태만 꺼내 쓰는 순환 층.

    PyTorch의 RNN/LSTM/GRU는 (출력 전체, 마지막 상태)를 돌려준다.
    Keras의 기본 동작(마지막 것만)과 맞추려면 이렇게 감싼다.
    """
    def __init__(self, kind, n_in, units, n_out):
        super().__init__()
        C = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}[kind]
        self.rnn = C(n_in, units, batch_first=True)
        self.head = nn.Linear(units, n_out)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.head(out[:, -1])              # 마지막 걸음만

def _train(model, xa, ya, xb, yb, loss_fn, lr, epochs, bs=64, cls=True):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    yt = torch.tensor(ya, dtype=torch.long if cls else torch.float32)
    if not cls:
        yt = yt.view(-1, 1)
    dl = DataLoader(TensorDataset(torch.tensor(xa, dtype=torch.float32), yt),
                    batch_size=bs, shuffle=True)
    for _ in range(dlbook.smoke.epochs(epochs)):
        model.train()
        for xx, yy in dl:
            xx, yy = xx.to(device), yy.to(device)
            opt.zero_grad(); loss_fn(model(xx), yy).backward(); opt.step()
    model.eval()
    with torch.no_grad():
        out = model(torch.tensor(xb, dtype=torch.float32).to(device)).cpu().numpy()
    return model, out

def train_seq(kind, sp, length, lr=0.001, seed=42, epochs=25):
    """분류: 표시된 자리의 부호 맞히기. (시험 정확도)

    이 함수 하나만 판마다 다르다. 아래의 모든 실험 셀은 세 판이 같다.
    """
    dlbook.set_seed(seed)
    if kind == "dnn":
        model = nn.Sequential(nn.Flatten(), nn.Linear(length * 2, 64),
                              nn.ReLU(), nn.Linear(64, 2))
    else:
        model = Recurrent(kind, 2, 32, 2)
    _, out = _train(model, sp.x_train, sp.y_train, sp.x_test, sp.y_test,
                    nn.CrossEntropyLoss(), lr, epochs)
    return metrics.accuracy(sp.y_test, out.argmax(1))

def train_forecast(kind, sp, lr=0.003, seed=42, epochs=30):
    """회귀: 다음 값 맞히기. (MAE, 파라미터 수)"""
    dlbook.set_seed(seed)
    length = sp.x_train.shape[1]
    if kind == "dnn":
        model = nn.Sequential(nn.Flatten(), nn.Linear(length, 64),
                              nn.ReLU(), nn.Linear(64, 1))
    else:
        model = Recurrent(kind, 1, 32, 1)
    model, out = _train(model, sp.x_train, sp.y_train, sp.x_test, sp.y_test,
                        nn.MSELoss(), lr, epochs, cls=False)
    n_params = sum(p.numel() for p in model.parameters())
    return metrics.mae(sp.y_test, out.reshape(-1)), n_params

## 10.3 실험 ① — 구조만 바꿔 봅니다

학습률을 0.001로 **고정**하고 구조만 바꿉니다.

In [ ]:
# 실험 ① — 학습률을 0.001로 **고정**하고 구조만 바꾼다.
lengths = [10, 80] if dlbook.smoke.is_smoke() else [10, 20, 40, 80]
kinds = ["dnn", "rnn", "lstm", "gru"]

print("학습률 0.001 고정, 시드 42 고정. 구조만 바꿉니다.")
print(f"{'길이':<8}" + "".join(f"{k:>11}" for k in kinds))
for L in lengths:
    xl, yl = data.memory_task(4000, length=L, seed=42)
    sl = data.split(xl, yl, val_ratio=0.15, test_ratio=0.15, seed=42)
    row = [train_seq(k, sl, L, lr=0.001, seed=42) for k in kinds]
    print(f"{L:<8}" + "".join(f"{v:>11.3f}" for v in row))
    for k, v in zip(kinds, row):
        dlbook.record(f"ch10_L{L}_{k}_lr0.001", v)

print()
print("⚠ 이 표만 보고 결론 내리면 틀립니다. 5장 §5.4에서 이미 겪은 함정입니다.")
print("  학습률을 고정해 놓고 구조만 바꿨습니다. 실험 ②를 보십시오.")

## 정리 — 그리고 경고

실험 ①의 표를 읽으면 *"길이 80에서 SimpleRNN이 LSTM보다 낫다"* 로 보입니다.

**교과서와 정반대이고, 그리고 이 결론은 틀렸습니다.**

학습률을 0.001로 **고정**해 놓고 구조만 바꿨기 때문입니다.
5장 §5.4에서 정확히 같은 실수를 했습니다.

**→ `ch10_lr.ipynb` 로 이어집니다.**

### 연습

1. 길이 160, 320으로 늘리면 어떻게 됩니까.
2. DNN이 길이 80에서도 0.99를 냅니다. **왜 그렇습니까.**
   (힌트: 표시 채널이 있습니다)
3. 표시 채널을 없애면 DNN이 어떻게 됩니까.